In [ ]:
import numpy as np
import random
from datasets import load_dataset
from PIL import Image
import os

# =======================================================================
# 🤖✨ AI 튜터의 미션: Tiny ImageNet 데이터셋 탐험하기! ✨🤖
# =======================================================================
# [데이터셋 제목] Tiny ImageNet 데이터셋
# [대략적인 의미] 작은 규모로 재구성된 ImageNet 데이터셋입니다.
# [설명] 이 데이터셋은 실제 이미지 분류(Image Classification)를 배우기 위해 사용됩니다.
#         만약 여러분이 AI 모델에게 "이 사진은 고양이일까? 아니면 강아지일까?"라고 질문한다면,
#         이 데이터셋은 '질문'과 '정답(Label)'을 짝지어 학습할 수 있는 귀한 재료가 됩니다!
#
# ✨ 오늘 목표: 데이터의 구조를 파헤치고, 전처리 과정의 맛보기 분석을 해보며,
# ✨ AI 데이터의 기초 체력을 다져봅시다! (feat. 스트리밍 데이터 로딩)
# =======================================================================

# --- 설정 상수 ---
DATASET_NAME = "zh-plus/tiny-imagenet"
SPLIT_NAME = "train"
SAMPLE_COUNT = 5  # 너무 많은 데이터를 로드하면 느려지니, 딱 5개만 볼게요!

print("🚀 데이터를 로드할 준비가 되었습니다! ✨")
print("🌟 친절한 코딩 튜터가 옆에서 도와드릴게요. 코드를 따라 와요! 🌟\n")

# 1. 데이터 로드 (스트리밍 최적화 및 에러 방지)
dataset = None
try:
    print(f"✅ 1단계: '{DATASET_NAME}' 데이터셋을 로드합니다. (분할: {SPLIT_NAME})")
    # 스트리밍 모드 시도 (메모리 효율 최고!)
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print("   👉 스트리밍 모드 로딩 성공! 메모리를 아끼며 데이터에 접근할 수 있어요.")
except Exception as e:
    print(f"   ⚠️ 스트리밍 로딩 실패 ({e}). 안전 모드로 전환합니다.")
    # 실패 시 일반 모드로 소량 다운로드 (메모리 부족 위험 감수)
    try:
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=False)
        print("   ✅ 일반 모드로 소량 데이터셋을 성공적으로 로드했습니다. 이제 진행할 수 있어요!")
    except Exception as e_fallback:
        print(f"   ❌ 치명적인 오류 발생: 데이터셋 로드에 실패했습니다. 오류: {e_fallback}")
        exit()

# 2. 데이터 샘플 확보 (len() 사용 금지 패턴 준수!)
# 스트리밍 여부와 관계없이 .take()를 사용하여 필요한 만큼만 가져옵니다.
if hasattr(dataset, "take"):
    print(f"\n🔍 2단계: {SAMPLE_COUNT}개의 샘플만 샘플링하여 분석을 시작할게요.")
    # take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    sample_iterator = dataset.take(SAMPLE_COUNT)
    # list()을 사용하여 반복 가능한 이터레이터를 확정된 리스트로 만듭니다.
    sample_data_list = list(sample_iterator)
else:
    # 일반 데이터셋 (Dataset)
    sample_data_list = list(dataset.take(SAMPLE_COUNT))

print("✨ 샘플 로드 완료! 이제 데이터 구조를 탐색해봅시다!")

# =======================================================================
# 💡 미션 1: 데이터 구조 훑어보기 (Exploratory Data Analysis)
# =======================================================================

print("\n" + "=".center(60, ' '))
print("🌟 미션 1: 데이터의 기본 구조 분석하기 (What is it?) 🌟".center(60, ' '))
print("=".center(60, ' '))

def explore_sample(sample_list):
    """샘플 리스트를 순회하며 데이터의 유형을 친절하게 분석합니다."""
    print(f"➡️ 샘플 {len(sample_list)}개를 순회하며 데이터 포인트를 확인합니다.")
    for i, sample in enumerate(sample_list):
        print(f"\n--- 🖼️ 샘플 {i+1} 분석 ---")
        
        # 1. 이미지 데이터 확인
        image = sample['image']
        print(f"   [IMAGE]: 데이터 타입은 <{type(image).__name__}>입니다. (보통 PIL Image 객체예요.)")
        
        # 2. 라벨 데이터 확인
        label_id = sample['label']
        print(f"   [LABEL ID]: 카테고리 ID는 '{label_id}'입니다.")

        # 3. 창의적 분석: 라벨 ID의 '흐름' 파악하기
        # tiny-imagenet은 무작위로 ID가 부여되어 있어요. 어떤 패턴이 숨어있을까요?
        label_numeric = label_id.replace('n', '').replace(':', '')
        print(f"   [Label 패턴]: ID를 숫자만으로 보면 '{label_numeric}'입니다. AI는 이 숫자 차이에서 패턴을 찾으려 해요!")

explore_sample(sample_data_list)


# =======================================================================
# 🔮 미션 2: 라벨 분포 분석 및 가상 예측 시뮬레이션
# =======================================================================

print("\n" + "=".center(60, ' '))
print("🧐 미션 2: 라벨 분포 및 가상 예측 시뮬레이션 (Guess the Category!) 🕵️")
print("=".center(60, ' '))

from collections import Counter

# 1. 전체 라벨 ID 수집 및 빈도 계산 (모든 샘플을 사용해 통계 계산)
all_labels = [sample['label'] for sample in sample_data_list]
label_counts = Counter(all_labels)

print("🧠 데이터셋의 핵심: 라벨 카운트 분석")
print("---------------------------------------")
print(f"🔍 분석된 샘플 {len(all_labels)}개에서 발견된 고유한 카테고리 종류: {len(label_counts)}가지.")
print("📊 카테고리별 등장 횟수 (Top 3):")
# 가장 많이 나온 3개의 라벨을 보여줍니다.
for label, count in label_counts.most_common(3):
    print(f"   -> 카테고리 '{label}'가 {count}번 등장했습니다. (가장 '핫'한 주제!)")

# 2. 가상 예측 함수 (AI 시뮬레이션)
def mystery_classifier_mock(sample):
    """
    입문자용 창의적 실습: 주어진 라벨 ID의 특징을 분석하여 가상의 예측 결과를 반환합니다.
    (실제 AI 학습은 필요하지만, 여기서는 재미로 구조를 분석해봅니다!)
    """
    label_id = sample['label']
    
    # 예시 규칙 1: ID의 길이가 10자 미만이면 '미스터리 오브제'
    if len(label_id) < 12:
        return "🔎 미스터리 오브제 (Mystery Object): ID 길이가 짧아 특정 범주를 추론하기 어려워요!"
    # 예시 규칙 2: 'n02'로 시작하는 ID는 '동물' 계열일 확률이 높다 (가정)
    elif label_id.startswith('n02'):
        return "🦁 동물류 (Animal Kingdom): 이 ID 패턴은 생명체와 관련 있을 가능성이 높아요!"
    # 그 외의 경우
    else:
        return "🌳 자연/사물군 (Nature/Manmade): 일반적인 사물이나 환경과 관련될 것 같아요."

print("\n🧠 AI 예측 시뮬레이션 실행:")
print("-----------------------------")

for i, sample in enumerate(sample_data_list):
    prediction = mystery_classifier_mock(sample)
    print(f"   [Sample {i+1}] -> {prediction}")

print("\n🎉✨ 끝! 🎉✨")
print("여러분, 정말 잘하셨어요! 데이터를 로드하고, 분석하고, 심지어 예측까지 시뮬레이션 해보았답니다!")
print("이처럼 AI는 데이터의 구조(형태)와 패턴(분포)을 이해하는 것에서부터 시작한답니다. 오늘도 파이썬 코딩 실력 만점이에요! 💯")